In [6]:
suppressPackageStartupMessages({
library(gplots)
library(NanoStringNCTools)
library(RColorBrewer)
library(grid)
library(gridExtra)
# library(ggalluvial)
library(knitr)
library(ggplot2)
library(dplyr)
library(plyr)
library(openxlsx)
library(limma)
library(DESeq2)
library(lme4)
library(AnnotationDbi)
library(org.Hs.eg.db)
library(clusterProfiler)
library(png)
library(grid)
library(gridExtra)
library(ggforce)
library(tidyverse)
library(GeomxTools)
library(tidyverse)
library(edgeR)
library(aliases2entrez)
library(biomaRt)
HGNC <- update_symbols()
library(stringr)
library(patchwork) 
source('/mnt/backup/Workspace/phd/geomx/r_scripts/geomx_helper_functions.R')
})

Fetching url...

Accessing data...

Rows: 49361 Columns: 6
── Column specification ──────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (5): Approved symbol, Status, Previous symbols, Alias symbols, Ensembl g...
dbl (1): NCBI Gene ID

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Checking validity...

done...



In [8]:
ontology_plot <- function(genes,title){
        # Convert gene symbols to Entrez IDs
        entrez_ids <- mapIds(org.Hs.eg.db, 
                             keys = genes, 
                             keytype = "SYMBOL", 
                             column = "ENTREZID")
        # Perform the GO enrichment analysis
        go_enrichment <- enrichGO(gene         = entrez_ids,
                                  OrgDb        = org.Hs.eg.db,  # Human
                                  ont          = "ALL",  # Biological process ontology
                                  pAdjustMethod = "BH",  # Benjamini & Hochberg adjustment method
                                  qvalueCutoff = 0.05)  # q-value cutoff

        # Print the results
        results_length <- nrow(as.data.frame(go_enrichment))
        if (results_length > 0){
            #p <- plot(barplot(go_enrichment, showCategory = 10,label_format=50, title=title)) + theme(axis.text.y=element_text(face="bold", size=8))
            p <- plot(dotplot(go_enrichment, showCategory = 10, title=title))
            return(p)
        } else {
            return(NULL)
        }
}

load("/mnt/backup/Workspace/crc/geomx_crc/geomx_crc.RData")
target_demoData$location <- ifelse(target_demoData$location == 'front' ,'border',target_demoData$location)
target_demoData$mucinous <- ifelse(pData(target_demoData)$mucinous == 'true','mucinous','non-mucinous')

# objectives

Check if in the mucinous group you have separated colon from breast or you have pooled everything mucinous.<br>
Next  step is to perform the analysis for mucinous breast.

- In breast: intra vs border/ intra vs extra /extra vs border
- Breast mucinous vs colon mucinous:  
    - CD68 (intra vs intra  and border vs border)
    - CK (intra vs intra)
    - CD45 (intra vs intra  and border vs border)
    
Have results in heat maps.<br>
Make deconvolution of CD68 segments using MoMac and output proportions (here re the metadata in the Excel file).

# breast:
1. intra vs border 
2. intra vs extra 
3. extra vs border

In [9]:
breast_subset <- target_demoData[,pData(target_demoData)$tissue == 'sein']
assayDataElement(object = breast_subset, elt = "log_q") <- assayDataApply(breast_subset, 2, FUN = log, base = 2, elt = "q_norm")
unique(pData(breast_subset)$location)
unique(pData(breast_subset)$mucinous)
unique(pData(breast_subset)$patient_id)
unique(pData(breast_subset)$tissue)

[1] "intra-tumor" "border"      "extra-tumor"

[1] "mucinous"

[1] "patient 9"  "patient 12" "patient 11" "patient 10"

[1] "sein"

In [ ]:
lmm_f <- function(geomx_dataset,test_variable,subsetting_variable){
    # convert test variables to factors        
    pData(geomx_dataset)$test_variable <- factor(pData(geomx_dataset)[[test_variable]])
    pData(geomx_dataset)$subsetting_variable <- factor(pData(geomx_dataset)[[subsetting_variable]])
    pData(geomx_dataset)[["slide"]] <- factor(pData(geomx_dataset)[["slide name"]])
    # run LMM:
    results <- c()
    for(state in levels(pData(geomx_dataset)$subsetting_variable)) {
        ind <- pData(geomx_dataset)$subsetting_variable == state
        if (length(unique(geomx_dataset[, ind][[test_variable]])) > 1){
            mixedOutmc <-
                mixedModelDE(geomx_dataset[, ind],
                             elt = "log_q",
                             modelFormula = ~ test_variable + (1 | slide),
                             groupVar = "test_variable",
                             nCores = 10,
                             multiCore = FALSE)

            # format results as data.frame
            r_test <- do.call(rbind, mixedOutmc["lsmeans", ])
            tests <- rownames(r_test)
            r_test <- as.data.frame(r_test)
            r_test$Contrast <- tests

            # use lapply in case you have multiple levels of your test factor to
            # correctly associate gene name with it's row in the results table
            r_test$Gene <-
                unlist(lapply(colnames(mixedOutmc),
                              rep, nrow(mixedOutmc["lsmeans", ][[1]])))
            r_test$Subset <- state
            r_test$FDR <- p.adjust(r_test$`Pr(>|t|)`, method = "fdr")
            r_test <- r_test[, c("Gene", "Subset", "Contrast", "Estimate","Pr(>|t|)", "FDR")]
            results <- rbind(results, r_test)
        } else {
            continue
        }
    }
    return(results)
}
breast_location_results <- lmm_f(breast_subset,test_variable = 'location',subsetting_variable = 'segment')

In [ ]:
unique(breast_location_results$Subset)
unique(breast_location_results$Contrast)

In [ ]:
volcano_simple_fn <- function(results,contrast) {
    results = results[results$Contrast == contrast,]
    pattern <- "^(.+?) - (.+)$"
    # Extract the words
    matches <- str_match(contrast, pattern)
    left_part <- trimws(matches[2])
    right_part <- trimws(matches[3])
    # Categorize Results based on P-value & FDR for plotting
    results$Color <- "NS or FC < 0.5"
    results$Color[results$`Pr(>|t|)` < 0.05] <- "P < 0.05"
    results$Color[results$FDR < 0.05] <- "FDR < 0.05"
    results$Color[results$FDR < 0.001] <- "FDR < 0.001"
    results$Color[abs(results$Estimate) < 0.5] <- "NS or FC < 0.5"
    results$Color <- factor(results$Color,levels = c("NS or FC < 0.5", "P < 0.05","FDR < 0.05", "FDR < 0.001"))

    # pick top genes for either side of volcano to label
    # order genes for convenience:
    results$invert_P <- (-log10(results$`Pr(>|t|)`)) * sign(results$Estimate)
    top_g <- c()
    for(cond in unique(results$Subset)) {
        ind <- results$Subset == cond
        neg_genes <- results[ind, ]$Gene[results[ind, ]$Estimate < 0 & results[ind, ]$FDR < 0.05]
        pos_genes <- results[ind, ]$Gene[results[ind, ]$Estimate > 0 & results[ind, ]$FDR < 0.05]

        top_g <- c(top_g,
                   results[ind, 'Gene'][
                   order(results[ind, 'invert_P'], decreasing = TRUE)[1:15]],
                   results[ind, 'Gene'][
                           order(results[ind, 'invert_P'], decreasing = FALSE)[1:15]])
    }
    top_g <- unique(top_g)
    results <- results[, -1*ncol(results)] # remove invert_P from matrix
    
    # Graph results
    volcano_plot <- ggplot(results,
           aes(x = Estimate, y = -log10(`Pr(>|t|)`),
               color = Color, label = Gene)) +
        geom_vline(xintercept = c(0.5, -0.5), lty = "dashed") +
        geom_hline(yintercept = -log10(0.05), lty = "dashed") +
        geom_point() +
        labs(x = paste('Enriched in',right_part,'<- log2(FC) -> Enriched in',left_part),y = "Significance, -log10(P)",color = "Significance") +
        scale_color_manual(values = c(`FDR < 0.001` = "dodgerblue",
                                      `FDR < 0.05` = "lightblue",
                                      `P < 0.05` = "orange2",
                                      `NS or FC < 0.5` = "gray"),
                           guide = guide_legend(override.aes = list(size = 4))) +
        scale_y_continuous(expand = expansion(mult = c(0,0.05))) +
        geom_text_repel(data = subset(results, Gene %in% top_g & FDR < 0.001),
                        size = 4, point.padding = 0.15, color = "black",
                        min.segment.length = .1, box.padding = .2,max.overlaps = 50) +
        theme_bw(base_size = 16) +
        theme(legend.position = "bottom") +
        facet_wrap(~Subset, scales = "free_y")
    
    
    volcano_plot <- volcano_plot + plot_annotation(title = contrast, theme = theme(plot.title = element_text(hjust = 0.5, size = 20)))
   
    return(volcano_plot)
}

options(repr.plot.width=16, repr.plot.height=8)
volcano_simple_fn(breast_location_results,"border - extra-tumor")
volcano_simple_fn(breast_location_results,"border - intra-tumor")
volcano_simple_fn(breast_location_results,"extra-tumor - intra-tumor")

In [ ]:
heatmap_fn <- function(geomx_dataset,result_df,subset,contrast,contrast_col,annotations,title,n){
    pattern <- "^(.+?) - (.+)$"
    # Extract the words
    matches <- str_match(contrast, pattern)
    left_part <- trimws(matches[2])
    right_part <- trimws(matches[3])
       
    geomx_dataset = geomx_dataset[,pData(geomx_dataset)$segment == subset & pData(geomx_dataset)[[contrast_col]] %in% c(left_part,right_part)]
    gap_col = ncol(geomx_dataset[,pData(geomx_dataset)[[contrast_col]] == left_part]) 

    results = result_df[result_df$Contrast == contrast & result_df$Subset == subset,]
    results$invert_P <- (-log10(results$`Pr(>|t|)`)) * sign(results$Estimate)
    gap_row = n
    
    extremes <- extreme_genes(results,n)

    GOI  <- c(extremes$pos,extremes$neg)
    data <- log2(assayDataElement(geomx_dataset[GOI,], elt = "q_norm"))
    data <- data[,order(pData(geomx_dataset)[[contrast_col]])]
    
    pheatmap(data,
             main = title,
             scale = "row", 
             show_rownames = T, show_colnames = F,
             border_color = NA,
             cluster_cols = F, cluster_rows = F,  
             gaps_col = c(gap_col),gaps_row = c(length(extremes$pos)),
             cutree_cols = 2, cutree_rows = 2,
             treeheight_row =0,treeheight_col = 0,
             breaks = seq(-3, 3, 0.05),
             legend = F, annotation_legend = T, annotation_names_row =T,fontsize_row=6, fontsize=6,
             color = colorRampPalette(c("navy",'white',"red"))(120),
             annotation_col = pData(geomx_dataset)[annotations]
            )
}
options(repr.plot.width=8, repr.plot.height=8)
heatmap_fn(breast_subset,breast_location_results,'CD68',"border - extra-tumor","location",c('patient_id','location'),'CD68 border vs extra-tumor',40)
heatmap_fn(breast_subset,breast_location_results,'CD68',"border - intra-tumor","location",c('patient_id','location'),'CD68 border vs intra-tumor',40)
heatmap_fn(breast_subset,breast_location_results,'CD68',"extra-tumor - intra-tumor","location",c('patient_id','location'),'CD68 intra vs extra-tumor',40)

heatmap_fn(breast_subset,breast_location_results,'CD45',"border - extra-tumor","location",c('patient_id','location'),'CD45 border vs extra-tumor',40)
heatmap_fn(breast_subset,breast_location_results,'CD45',"border - intra-tumor","location",c('patient_id','location'),'CD45 border vs intra-tumor',40)
heatmap_fn(breast_subset,breast_location_results,'CD45',"extra-tumor - intra-tumor","location",c('patient_id','location'),'CD45 intra vs extra-tumor',40)

- Breast mucinous vs colon mucinous:  
    - CD68 (intra vs intra  and border vs border)
    - CK (intra vs intra)
    - CD45 (intra vs intra  and border vs border)

In [ ]:
mucinous_intra_subset = target_demoData[, pData(target_demoData)$mucinous == 'mucinous' & pData(target_demoData)$tissue %in% c('sein','colon') & pData(target_demoData)$location == 'intra-tumor']
assayDataElement(object = mucinous_intra_subset, elt = "log_q") <- assayDataApply(mucinous_intra_subset, 2, FUN = log, base = 2, elt = "q_norm")
mucinous_intra_tissue_results <- lmm_f(mucinous_intra_subset,test_variable = 'tissue',subsetting_variable = 'segment')

In [ ]:
options(repr.plot.width=24, repr.plot.height=8)
volcano_simple_fn(mucinous_intra_tissue_results,'colon - sein')
options(repr.plot.width=8, repr.plot.height=8)
heatmap_fn(mucinous_intra_subset,mucinous_intra_tissue_results,'CK',"colon - sein",'tissue',c('tissue','patient_id'),"CK intra sein - intra colon",40)
heatmap_fn(mucinous_intra_subset,mucinous_intra_tissue_results,'CD68',"colon - sein",'tissue',c('tissue','patient_id'),"CD68 intra sein - intra colon",40)
heatmap_fn(mucinous_intra_subset,mucinous_intra_tissue_results,'CD45',"colon - sein",'tissue',c('tissue','patient_id'),"CD45 intra sein - intra colon",40)

In [ ]:
unique(pData(mucinous_intra_subset)$location)
unique(pData(mucinous_intra_subset)$mucinous)
unique(pData(mucinous_intra_subset)$patient_id)
unique(pData(mucinous_intra_subset)$tissue)

In [ ]:
mucinous_border_subset = target_demoData[, pData(target_demoData)$mucinous == 'mucinous' & pData(target_demoData)$tissue %in% c('sein','colon') & pData(target_demoData)$location == 'border']
assayDataElement(object = mucinous_border_subset, elt = "log_q") <- assayDataApply(mucinous_border_subset, 2, FUN = log, base = 2, elt = "q_norm")
mucinous_border_tissue_results <- lmm_f(mucinous_border_subset,
                                        test_variable = 'tissue',
                                        subsetting_variable = 'segment')

In [ ]:
options(repr.plot.width=16, repr.plot.height=8)
volcano_simple_fn(mucinous_border_tissue_results,'colon - sein')
options(repr.plot.width=8, repr.plot.height=8)
heatmap_fn(mucinous_border_subset,mucinous_border_tissue_results,'CD68',"colon - sein",'tissue',c('tissue','patient_id'),"CD68 border colon - border sein",40)
heatmap_fn(mucinous_border_subset,mucinous_border_tissue_results,'CD45',"colon - sein",'tissue',c('tissue','patient_id'),"CD45 border colon - border sein",40)

In [ ]:
unique(pData(mucinous_border_subset)$location)
unique(pData(mucinous_border_subset)$mucinous)
unique(pData(mucinous_border_subset)$patient_id)
unique(pData(mucinous_border_subset)$tissue)